In [ ]:
# 02 Preprocessing

In diesem Notebook werden die NYC Yellow Taxi Rohdaten vorbereitet.

Ziel:
- relevante Spalten auswählen
- Datentypen über alle Jahre vereinheitlichen
- Jahr und Monat aus dem Dateinamen ergänzen
- zusätzliche Spalten für die spätere Analyse berechnen
- vorbereiteten Datensatz in HDFS speichern

Die Datenqualität wird in diesem Schritt noch nicht bereinigt. Ungültige oder unplausible Werte werden erst im nächsten Notebook `03_data_quality` entfernt.

In [ ]:
Start Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce

spark = SparkSession.builder \
    .appName("Taxi Preprocessing") \
    .getOrCreate()

spark

26/05/19 14:21:40 WARN Utils: Your hostname, bdlc-021 resolves to a loopback address: 127.0.1.1; using 10.176.129.84 instead (on interface ens3)
26/05/19 14:21:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 14:21:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/19 14:21:59 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
Pfade definieren

In [2]:
RAW_PATH = "/taxi/raw"
PREPARED_PATH = "/taxi/prepared"

YEARS = list(range(2015, 2022))
MONTHS = [f"{month:02d}" for month in range(1, 13)]

print("Raw path:", RAW_PATH)
print("Prepared path:", PREPARED_PATH)
print("Years:", YEARS)
print("Months:", MONTHS)

Raw path: /taxi/raw
Prepared path: /taxi/prepared
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021]
Months: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']


In [ ]:
Relevante Spalten wählen

In [3]:
selected_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount"
]

In [ ]:
Monatsdatei einlesen und Vereinheitlichen

In [4]:
def read_and_prepare_month(year: int, month: str):
    path = f"{RAW_PATH}/{year}/yellow_tripdata_{year}-{month}.parquet"
    
    df = spark.read.parquet(path)
    
    df_prepared = df.select(
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        F.col("passenger_count").cast("double").alias("passenger_count"),
        F.col("trip_distance").cast("double").alias("trip_distance_miles"),
        F.col("PULocationID").cast("long").alias("pickup_location_id"),
        F.col("DOLocationID").cast("long").alias("dropoff_location_id"),
        F.col("payment_type").cast("long").alias("payment_type"),
        F.col("fare_amount").cast("double").alias("fare_amount"),
        F.col("tip_amount").cast("double").alias("tip_amount"),
        F.col("total_amount").cast("double").alias("total_amount")
    )
    
    df_prepared = df_prepared \
        .withColumn("file_year", F.lit(year)) \
        .withColumn("file_month", F.lit(int(month))) \
        .withColumn("pickup_year", F.year("pickup_datetime")) \
        .withColumn("pickup_month", F.month("pickup_datetime")) \
        .withColumn("pickup_hour", F.hour("pickup_datetime")) \
        .withColumn("pickup_weekday", F.dayofweek("pickup_datetime")) \
        .withColumn(
            "trip_duration_minutes",
            (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
        ) \
        .withColumn(
            "trip_distance_km",
            F.col("trip_distance_miles") * 1.60934
        ) \
        .withColumn(
            "avg_speed_kmh",
            F.col("trip_distance_km") / (F.col("trip_duration_minutes") / 60)
        ) \
        .withColumn(
            "tip_percentage",
            F.when(
                F.col("total_amount") > 0,
                F.col("tip_amount") / F.col("total_amount") * 100
            ).otherwise(None)
        )
    
    return df_prepared

In [ ]:
Funktionstest mit einem Monat

In [5]:
test_df = read_and_prepare_month(2021, "09")

test_df.printSchema()
test_df.show(5, truncate=False)

root
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance_miles: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- file_year: integer (nullable = false)
 |-- file_month: integer (nullable = false)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_weekday: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- tip_percentage: double (nullable = true)



+-------------------+-------------------+---------------+-------------------+------------------+-------------------+------------+-----------+----------+------------+---------+----------+-----------+------------+-----------+--------------+---------------------+------------------+------------------+------------------+
|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance_miles|pickup_location_id|dropoff_location_id|payment_type|fare_amount|tip_amount|total_amount|file_year|file_month|pickup_year|pickup_month|pickup_hour|pickup_weekday|trip_duration_minutes|trip_distance_km  |avg_speed_kmh     |tip_percentage    |
+-------------------+-------------------+---------------+-------------------+------------------+-------------------+------------+-----------+----------+------------+---------+----------+-----------+------------+-----------+--------------+---------------------+------------------+------------------+------------------+
|2021-09-01 06:04:34|2021-09-01 06:15:28|2.0  

In [ ]:
Test Count

In [6]:
print("Test records:", test_df.count())

Test records: 2963793


In [ ]:
Alle Dateien einlesen

In [7]:
prepared_dfs = []

for year in YEARS:
    for month in MONTHS:
        print(f"Reading {year}-{month}")
        prepared_dfs.append(read_and_prepare_month(year, month))

print("Number of monthly DataFrames:", len(prepared_dfs))

Reading 2015-01
Reading 2015-02
Reading 2015-03
Reading 2015-04
Reading 2015-05
Reading 2015-06
Reading 2015-07
Reading 2015-08
Reading 2015-09
Reading 2015-10
Reading 2015-11
Reading 2015-12
Reading 2016-01
Reading 2016-02
Reading 2016-03
Reading 2016-04
Reading 2016-05
Reading 2016-06
Reading 2016-07
Reading 2016-08
Reading 2016-09
Reading 2016-10
Reading 2016-11
Reading 2016-12
Reading 2017-01
Reading 2017-02
Reading 2017-03
Reading 2017-04
Reading 2017-05
Reading 2017-06
Reading 2017-07
Reading 2017-08
Reading 2017-09
Reading 2017-10
Reading 2017-11
Reading 2017-12
Reading 2018-01
Reading 2018-02
Reading 2018-03
Reading 2018-04
Reading 2018-05
Reading 2018-06
Reading 2018-07
Reading 2018-08
Reading 2018-09
Reading 2018-10
Reading 2018-11
Reading 2018-12
Reading 2019-01
Reading 2019-02
Reading 2019-03
Reading 2019-04
Reading 2019-05
Reading 2019-06
Reading 2019-07
Reading 2019-08
Reading 2019-09
Reading 2019-10
Reading 2019-11
Reading 2019-12
Reading 2020-01
Reading 2020-02
Reading 

In [ ]:
Zusammenführen

In [8]:
df_prepared = reduce(lambda df1, df2: df1.unionByName(df2), prepared_dfs)

df_prepared.printSchema()

root
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance_miles: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- file_year: integer (nullable = false)
 |-- file_month: integer (nullable = false)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_weekday: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- tip_percentage: double (nullable = true)



In [ ]:
Records prüfen

In [9]:
prepared_count = df_prepared.count()

print(f"Prepared records: {prepared_count}")

[Stage 91:===================================================> (886 + 12) / 914]

Prepared records: 633694594


In [ ]:
Spalten überprüfen

In [10]:
df_prepared.select(
    "file_year",
    "file_month",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_hour",
    "pickup_weekday",
    "trip_duration_minutes",
    "trip_distance_km",
    "avg_speed_kmh",
    "tip_percentage"
).show(10, truncate=False)

26/05/19 14:25:48 WARN DAGScheduler: Broadcasting large task binary with size 1124.1 KiB
26/05/19 14:25:48 WARN DAGScheduler: Broadcasting large task binary with size 1124.1 KiB
26/05/19 14:25:48 WARN DAGScheduler: Broadcasting large task binary with size 1124.1 KiB


+---------+----------+-------------------+-------------------+-----------+--------------+---------------------+------------------+------------------+------------------+
|file_year|file_month|pickup_datetime    |dropoff_datetime   |pickup_hour|pickup_weekday|trip_duration_minutes|trip_distance_km  |avg_speed_kmh     |tip_percentage    |
+---------+----------+-------------------+-------------------+-----------+--------------+---------------------+------------------+------------------+------------------+
|2015     |1         |2015-01-01 00:11:33|2015-01-01 00:16:48|0          |5             |5.25                 |1.60934           |18.392457142857143|16.666666666666664|
|2015     |1         |2015-01-01 00:18:24|2015-01-01 00:24:20|0          |5             |5.933333333333334    |1.448406          |14.646802247191012|0.0               |
|2015     |1         |2015-01-01 00:26:19|2015-01-01 00:41:06|0          |5             |14.783333333333333   |5.63269           |22.860974069898536|16.666

In [ ]:
Schema kontrollieren

In [11]:
schema_check = spark.createDataFrame(
    [(field.name, field.dataType.simpleString(), field.nullable) for field in df_prepared.schema.fields],
    ["column_name", "data_type", "nullable"]
)

schema_check.show(50, truncate=False)

+---------------------+---------+--------+
|column_name          |data_type|nullable|
+---------------------+---------+--------+
|pickup_datetime      |timestamp|true    |
|dropoff_datetime     |timestamp|true    |
|passenger_count      |double   |true    |
|trip_distance_miles  |double   |true    |
|pickup_location_id   |bigint   |true    |
|dropoff_location_id  |bigint   |true    |
|payment_type         |bigint   |true    |
|fare_amount          |double   |true    |
|tip_amount           |double   |true    |
|total_amount         |double   |true    |
|file_year            |int      |false   |
|file_month           |int      |false   |
|pickup_year          |int      |true    |
|pickup_month         |int      |true    |
|pickup_hour          |int      |true    |
|pickup_weekday       |int      |true    |
|trip_duration_minutes|double   |true    |
|trip_distance_km     |double   |true    |
|avg_speed_kmh        |double   |true    |
|tip_percentage       |double   |true    |
+----------

In [ ]:
Anzahl prüfen

In [12]:
df_prepared.groupBy("file_year", "file_month") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("file_year", "file_month") \
    .show(100)

[Stage 100:====================================================>(912 + 2) / 914]

+---------+----------+------------+
|file_year|file_month|record_count|
+---------+----------+------------+
|     2015|         1|    12741035|
|     2015|         2|    12442394|
|     2015|         3|    13342951|
|     2015|         4|    13063758|
|     2015|         5|    13157677|
|     2015|         6|    12324936|
|     2015|         7|    11559666|
|     2015|         8|    11123123|
|     2015|         9|    11218122|
|     2015|        10|    12307333|
|     2015|        11|    11305240|
|     2015|        12|    11452996|
|     2016|         1|    10905067|
|     2016|         2|    11375412|
|     2016|         3|    12203824|
|     2016|         4|    11927996|
|     2016|         5|    11832049|
|     2016|         6|    11131645|
|     2016|         7|    10294080|
|     2016|         8|     9942263|
|     2016|         9|    10116018|
|     2016|        10|    10854626|
|     2016|        11|    10102128|
|     2016|        12|    10446697|
|     2017|         1|     9

In [ ]:
Dataset nach HDFS speichern

In [16]:
#Fehlerbehebung: Aufgrund von falschen Zeitstempeln wird das schreiben blockiert

spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")




In [18]:
import subprocess

subprocess.run("hdfs dfs -rm -r -f /taxi/prepared", shell=True)

Deleted /taxi/prepared


CompletedProcess(args='hdfs dfs -rm -r -f /taxi/prepared', returncode=0)

In [19]:
df_prepared.write \
    .mode("overwrite") \
    .partitionBy("file_year", "file_month") \
    .parquet(PREPARED_PATH)

26/05/19 14:34:44 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
                                                                                

In [ ]:
Kontrolle

In [21]:
import subprocess

command = "hdfs dfs -du -h -s /taxi/prepared"
result = subprocess.check_output(command, shell=True).decode()

print(result)

18.2 G  36.4 G  /taxi/prepared



In [ ]:
Spark stoppen

In [22]:
spark.stop()